In [19]:
# Install required packages
!pip install --upgrade androguard pyaxmlparser opendatasets

# Import libraries
from pyaxmlparser import APK
import pandas as pd
import pickle
import opendatasets as od
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Download the dataset from Kaggle
od.download('https://www.kaggle.com/datasets/ankit1743/android-malware-detection-dataset')

# Load the dataset
data = pd.read_csv('/content/android-malware-detection-dataset/Android_Malware.csv')

# Prepare features and labels
X = data.drop('Result', axis=1)
y = data['Result']
PERMISSION_LIST = list(X.columns)

# Split dataset into training and testing sets for proper evaluation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train the Random Forest model on training data
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate on test data
print("Test accuracy:", model.score(X_test, y_test))

# Save the trained model
with open('rf_model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Load the model
model = pickle.load(open('rf_model.pkl', 'rb'))

# Function to extract permissions from APK and build feature vector
def extract_permissions_pyaxml(apk_path):
    apk = APK(apk_path)
    perms = apk.get_permissions()
    perms_vector = [1 if perm in perms else 0 for perm in PERMISSION_LIST]
    return perms_vector

# Updated scan function with adjustable malware probability threshold
def scan_apk_pyaxml(apk_path, malware_probability_threshold=0.2):
    try:
        features = extract_permissions_pyaxml(apk_path)
        features_df = pd.DataFrame([features], columns=PERMISSION_LIST)
        prob = model.predict_proba(features_df)[0]
        print("Probability (Benign, Malware):", prob)
        if prob[1] >= malware_probability_threshold:
            print(f"Malware detected! Malware probability {prob[1]:.2f} ≥ threshold {malware_probability_threshold}")
        else:
            print(f"App is benign. Malware probability {prob[1]:.2f} < threshold {malware_probability_threshold}")
    except Exception as e:
        print("Error scanning APK:", e)

# For uploading APK file in Colab, run this in the notebook:
from google.colab import files
uploaded = files.upload()  # Upload your APK file here

# Example scan call (update filename accordingly)
scan_apk_pyaxml("/content/wildfire-test-apk-file.apk", malware_probability_threshold=0.2)


Skipping, found downloaded files in "./android-malware-detection-dataset" (use force=True to force download)
Test accuracy: 0.97125


Probability (Benign, Malware): [0.78907362 0.21092638]
Malware detected! Malware probability 0.21 ≥ threshold 0.2


In [20]:
# Example scan call (update filename accordingly)
scan_apk_pyaxml("/content/instagram.apk", malware_probability_threshold=0.2)

Probability (Benign, Malware): [0.95 0.05]
App is benign. Malware probability 0.05 < threshold 0.2
